# Lezione 4 — Reti convoluzionali (CNN) con Keras

## Obiettivi di apprendimento
- Capire perché una rete convoluzionale (CNN) è più adatta di una rete
  densa per lavorare con le immagini.
- Costruire una CNN con uno strato `Conv2D` + `MaxPooling2D` e confrontarne i
  risultati con la rete densa della lezione precedente.
- Costruire una seconda CNN più profonda, con due cicli di convoluzione e
  pooling.
- Capire cos'è il `batch_size` e come influisce sull'addestramento.

## Cosa costruiremo
Due architetture di rete convoluzionale addestrate sullo stesso dataset
MNIST della lezione precedente, per poter confrontare direttamente "rete densa
vs. CNN" e "CNN semplice vs. CNN più profonda".

**Nota sui tempi di esecuzione:** il materiale originale di IBM addestra
questi modelli con `epochs=10`. In questo notebook usiamo **3 epoche** affinché
la lezione giri in un tempo ragionevole su un Colab con CPU (qualche
minuto invece di decine di minuti). Se vuoi riprodurre i
risultati di precisione riportati nel manuale (che si aggirano sul 98-99%),
puoi alzare `epochs` a 10 quando ti eserciti per conto tuo, soprattutto
se attivi una GPU su Colab (`Ambiente di esecuzione → Cambia tipo di
ambiente di esecuzione → GPU`).

Keras e TensorFlow sono già preinstallati su Colab: non serve nessun
`!pip install` per questa lezione.

## Ripasso veloce (capitolo 7 del manuale)

Una CNN combina tre tipi di strati:
- **`Conv2D`**: fa scorrere piccoli filtri sull'immagine per rilevare
  pattern locali (bordi, curve), riutilizzando lo stesso filtro su tutta
  l'immagine.
- **`MaxPooling2D`**: riduce la dimensione delle mappe di caratteristiche
  mantenendo il valore massimo di ogni piccola regione.
- **`Flatten`**: appiattisce il risultato prima di passarlo agli strati `Dense`
  normali, che eseguono la classificazione finale con softmax.

A differenza della rete densa della lezione precedente, qui le immagini
**non** vengono appiattite in ingresso: restano nel formato
`(altezza, larghezza, canali)`.

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.layers import Conv2D        # strati convoluzionali
from keras.layers import MaxPooling2D  # strati di pooling (raggruppamento)
from keras.layers import Flatten       # appiattire prima degli strati densi
from keras.utils import to_categorical
from keras.datasets import mnist

# MNIST è incluso in Keras, come nella lezione precedente.
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Reshape in formato immagine [campioni][altezza][larghezza][canali]. A differenza
# della rete densa, qui NON appiattiamo: manteniamo la struttura spaziale.
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1).astype('float32')
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1).astype('float32')

# Normalizziamo i pixel e codifichiamo le etichette in one-hot, come
# nella lezione precedente.
X_train = X_train / 255
X_test = X_test / 255
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
num_classes = y_test.shape[1]

print("X_train:", X_train.shape)
print("Numero di classi:", num_classes)

## CNN 1 — Un solo set di convoluzione + pooling

16 filtri 5x5 nello strato convoluzionale, seguiti da un pooling 2x2, e
poi gli stessi strati densi della rete di classificazione della lezione
precedente.

In [ ]:
def convolutional_model():
    model = Sequential()
    model.add(Input(shape=(28, 28, 1)))
    model.add(Conv2D(16, (5, 5), strides=(1, 1), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = convolutional_model()
model.summary()

In [ ]:
# NOTA: il materiale originale di IBM usa epochs=10. Qui usiamo 3 epoche
# affinché la lezione giri in pochi minuti su un Colab con CPU; alza a 10
# se vuoi riprodurre la precisione esatta riportata nel manuale.
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=3, batch_size=200, verbose=2)

scores = model.evaluate(X_test, y_test, verbose=0)
print("Precisione: {:.4f} \n Errore: {:.4f}".format(scores[1], 100 - scores[1] * 100))

## CNN 2 — Due set di convoluzione + pooling

Ora aggiungiamo un secondo ciclo di `Conv2D` + `MaxPooling2D` prima di
appiattire, in modo che la rete possa combinare i pattern semplici rilevati nel
primo ciclo in caratteristiche più astratte.

In [ ]:
def convolutional_model_v2():
    model = Sequential()
    model.add(Input(shape=(28, 28, 1)))
    model.add(Conv2D(16, (5, 5), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

    # secondo ciclo: estrae caratteristiche più astratte a partire da
    # quelle già rilevate dal primo ciclo
    model.add(Conv2D(8, (2, 2), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model_v2 = convolutional_model_v2()
model_v2.summary()

In [ ]:
# Come prima: 3 epoche invece delle 10 del materiale originale, per
# tempi di esecuzione ragionevoli in lezione.
model_v2.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=3, batch_size=200, verbose=2)

scores_v2 = model_v2.evaluate(X_test, y_test, verbose=0)
print("Precisione: {:.4f} \n Errore: {:.4f}".format(scores_v2[1], 100 - scores_v2[1] * 100))

## Riepilogo della lezione

- Una CNN elabora le immagini nel loro formato originale `(altezza, larghezza, canali)`,
  senza appiattirle in ingresso, a differenza di una rete densa.
- `Conv2D` rileva pattern locali riutilizzando lo stesso filtro su tutta
  l'immagine; `MaxPooling2D` riduce la dimensionalità mantenendo i
  valori massimi; `Flatten` appiattisce prima degli strati densi finali.
- Impilare più di un ciclo di convoluzione + pooling permette di imparare
  caratteristiche più astratte.
- `batch_size` controlla quanti campioni vede la rete prima di aggiornare i
  suoi pesi una volta; nella pratica di questa lezione sperimenterai
  cambiandolo.
- Con sole 3 epoche si vedono già risultati solidi (sopra il 95% di
  precisione); con le 10 epoche del materiale originale la precisione sale
  ancora di più, tipicamente sopra il 98%.

**Prosegui con:**
- Capitolo del manuale: [`07_cnn.md`](../manual/07_cnn_IT.md)
- Notebook di pratica di questa lezione: [`clase04_practica_cnn.ipynb`](../notebooks_practica/clase04_practica_cnn_IT.ipynb)